# Chest X-ray Classification: End-to-End Pipeline
### Final Project Report - Group 5

This notebook represents the entire body of work performed in this project, from raw data preparation and self-supervised pre-training to final supervised fine-tuning and model interpretation.

## 1. Executive Summary

### i. What worked best and why?
The CLAHE preprocessing made a big difference. This allows the model to detect faint pathological features without amplifying background noise. The loss function that we had the most success with was the asymmetric loss function, designed to handle the heavy class imbalance in the NIH dataset. Another major success was embedding the view position (PA/AP), which allowed the model to adjust its feature recognition based on the anatomical perspective.

### ii. What didn't help?
Mixup augmentation, horizontal flips, and progressive resizing did not yield improvements. Mixup created unrealistic medical samples, while horizontal flips were counter-productive since the human anatomy is not symmetrical in X-rays. Progressive resizing lost critical fine-grained details necessary for detecting pathology.

### iii. Final Model Architecture
We utilize a **Swin Transformer V2** backbone, pre-trained using **SimMIM** (Self-Supervised Learning). The architecture is enhanced with a custom MLP head, **class-specific attention pooling**, and **view position embeddings**.

## 2. Environment Setup & Constants

In [ ]:
import os
import glob
import math
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, ExifTags
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, multilabel_confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer
from skmultilearn.model_selection import IterativeStratification
from torch.amp import autocast

# Backbone Architecture (Imported from local implementation)
from swin_transformer_v2 import SwinTransformerV2

# Constants
ALL_CLASSES = [
    "Atelectasis","Cardiomegaly","Consolidation","Edema",
    "Effusion","Emphysema","Fibrosis","Hernia",
    "Infiltration","Mass","No Finding","Nodule",
    "Pleural_Thickening","Pneumonia","Pneumothorax",
]

NIH_CXR8_CUSTOM_MEAN = [0.5249, 0.5249, 0.5249]
NIH_CXR8_CUSTOM_STD  = [0.2622, 0.2622, 0.2622]

CLIP_LIMIT = 1.5
TILE_GRID_SIZE = 4

## 3. Phase 1: Data Preparation & Offline Preprocessing

Before training, we calculated the dataset-wide mean and standard deviation and performed a batch-preprocessing step to normalize orientation and apply CLAHE for contrast enhancement.

In [ ]:
def normalize_cxr_image(img):
    """
    Handles EXIF orientation, lateral detection, and rib-gradient checks 
    to ensure all images are upright and PA/AP oriented.
    """
    if img.mode != "L":
        img = img.convert("L")

    arr = np.array(img)
    h, w = arr.shape

    # Lateral detection (skipped if too bright at edges)
    left_edge, right_edge = arr[:, :int(w * 0.15)].mean(), arr[:, int(w * 0.85):].mean()
    if max(left_edge, right_edge) > arr.mean() * 1.35: return None

    # Upside-down detection
    if arr[:h//3].mean() > arr[-h//3:].mean() * 1.10: arr = np.flipud(arr)
    
    # Rib gradient check
    arr_f = arr.astype(np.float32)
    sobel_y = cv2.Sobel(arr_f, cv2.CV_32F, 0, 1, ksize=5)
    if np.abs(sobel_y[:h//2]).mean() < np.abs(sobel_y[h//2:]).mean() * 0.85: arr = np.flipud(arr)

    return arr.astype(np.uint8)

def apply_batch_clahe(src_root, dst_root):
    """
    Logic used in preprocess.py to prepare the dataset for training.
    """
    clahe = cv2.createCLAHE(clipLimit=CLIP_LIMIT, tileGridSize=(TILE_GRID_SIZE, TILE_GRID_SIZE))
    # ... Iteration logic with multiprocessing ...
    # Example for one image:
    # img_normalized = normalize_cxr_image(Image.open(path))
    # if img_normalized is not None: 
    #     img_enhanced = clahe.apply(img_normalized)
    #     Image.fromarray(img_enhanced).save(dst_path)

## 4. Phase 2: Self-Supervised Learning (SimMIM Stage)

To better initialize our model for medical imagery, we used **SimMIM** (Simple Masked Image Modeling). This teaches the backbone to reconstruct masked-out patches of X-ray images, forcing it to learn anatomical structures without labels.

In [ ]:
class SimMIM_SwinV2(nn.Module):
    """
    Reconstructs pixel patches at masked positions to learn visual representations.
    """
    def __init__(self, backbone, img_size=256, patch_size=4, mask_ratio=0.6):
        super().__init__()
        self.backbone = backbone
        self.mask_ratio = mask_ratio
        self.patch_size = patch_size
        self.decoder = nn.Sequential(
            nn.Conv2d(backbone.num_features, backbone.num_features, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(backbone.num_features, 3 * patch_size * patch_size, kernel_size=1),
        )

    def forward(self, imgs):
        B, L = imgs.size(0), (imgs.size(2) // self.patch_size) ** 2
        # Masking logic (e.g., 60% of patches zeroed out)
        # ...
        # Backbone encoding
        # x_feat = self.backbone(imgs_masked)
        # Upsample & Reconstruct
        # pred = self.decoder(x_up)
        # loss = F.l1_loss(pred_masked, target_masked)
        pass

## 5. Phase 3: Supervised Fine-Tuning

In the final stage, we fine-tune the SimMIM-pretrained SwinV2 backbone on the labeled NIH CXR8 dataset using a custom multi-label head.

In [ ]:
class ClassSpecificAttnPool(nn.Module):
    """Learns a unique spatial attention query for every disease category."""
    def __init__(self, C, num_classes):
        super().__init__()
        self.norm = nn.LayerNorm(C)
        self.query = nn.Linear(C, num_classes, bias=False)
        self.temp = nn.Parameter(torch.ones(1))

    def forward(self, feats):
        feats = self.norm(feats)
        attn = torch.softmax(self.query(feats) / self.temp.clamp(min=0.1), dim=1)
        return torch.einsum("bnc,bnk->bkc", feats, attn)

class SwinWithView(nn.Module):
    """Final Model: Backbone + View Embedding + Class-Specific Attention."""
    def __init__(self, backbone, num_classes):
        super().__init__()
        C = backbone.num_features
        self.backbone = backbone
        self.attn_pool = ClassSpecificAttnPool(C, num_classes)
        self.view_embed = nn.Embedding(2, 32)
        self.view_mlp = nn.Sequential(nn.Linear(32, 128), nn.GELU(), nn.Linear(128, C * 2))
        self.view_scale = nn.Parameter(torch.tensor(0.35))
        self.head = nn.Sequential(nn.LayerNorm(C), nn.Linear(C, 512), nn.GELU(), nn.Linear(512, 1))

    def forward(self, x, view_id):
        feats = self.backbone.forward_features(x)
        v = self.view_mlp(self.view_embed(view_id))
        gamma, beta = v.chunk(2, dim=-1)
        feats = feats * (1 + torch.sigmoid(self.view_scale) * 2.0 * gamma.unsqueeze(1)) + beta.unsqueeze(1)
        pooled = self.attn_pool(feats)
        return self.head(pooled.reshape(-1, pooled.size(-1))).reshape(x.size(0), -1)

## 6. Phase 4: Evaluation & Performance Metrics

We evaluate the model using Mean AUC across all classes and visualize the training dynamics.

In [ ]:
def plot_learning_curves(log_csv_path):
    """Plots training and validation metrics."""
    df = pd.read_csv(log_csv_path)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    sns.lineplot(data=df, x='epoch', y='tr_loss', label='Train Loss', ax=axes[0])
    sns.lineplot(data=df, x='epoch', y='val_loss', label='Val Loss', ax=axes[0])
    axes[0].set_title('Loss Curves')

    sns.lineplot(data=df, x='epoch', y='tr_auc', label='Train AUC', ax=axes[1])
    sns.lineplot(data=df, x='epoch', y='val_auc', label='Val AUC', ax=axes[1])
    axes[1].set_title('AUC Curves')
    plt.show()

def plot_roc_curves(labels, probs):
    """Generates ROC curves for each disease category."""
    plt.figure(figsize=(10, 8))
    for i, cls in enumerate(ALL_CLASSES):
        if cls == "No Finding": continue
        fpr, tpr, _ = roc_curve(labels[:, i], probs[:, i])
        auc = roc_auc_score(labels[:, i], probs[:, i])
        plt.plot(fpr, tpr, label=f'{cls} (AUC={auc:.2f})')
    
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Multi-label ROC Curves')
    plt.legend(loc='lower right', fontsize='small', ncol=2)
    plt.show()

## 7. Phase 5: Model Interpretability (GradCAM)

To build trust in the model, we use GradCAM to ensure the network is focusing on relevant pathological regions (e.g., lungs for pneumonia, heart for cardiomegaly).

In [ ]:
class GradCAM:
    def __init__(self, model, device):
        self.model, self.device = model, device
        self._feats, self._grads = None, None
        target = model.backbone.layers[-1]
        self._fwd_hook = target.register_forward_hook(lambda m, i, o: setattr(self, '_feats', o[0] if isinstance(o, tuple) else o))
        self._bwd_hook = target.register_full_backward_hook(lambda m, gi, go: setattr(self, '_grads', go[0]))

    def __call__(self, img_tensor, class_idx, view_id=0):
        self.model.eval()
        x, v = img_tensor.unsqueeze(0).to(self.device), torch.tensor([view_id], device=self.device)
        logits = self.model(x, v)
        self.model.zero_grad()
        logits[0, class_idx].backward()
        weights = self._grads.detach().mean(dim=1, keepdim=True)
        cam = torch.relu((weights * self._feats.detach()).sum(dim=-1)).squeeze(0)
        cam = cam.reshape(int(cam.shape[0]**0.5), -1).cpu().numpy()
        return (cam - cam.min()) / (cam.max() + 1e-8)

def visualize_interpretability(model, img_tensor, img_np, class_idx, device, gradcam):
    cam = gradcam(img_tensor, class_idx)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_np, cmap='gray'); axes[0].set_title("Original X-ray")
    axes[1].imshow(img_np, cmap='gray'); axes[1].imshow(heatmap, alpha=0.45)
    axes[1].set_title(f"Model Focus: {ALL_CLASSES[class_idx]}")
    plt.show()